# Python and CLI for Data Science - Session 08

- *Course*: Python and CLI for Data Science
- *Session*: 08
- *Unit*: Pandas Aggregation

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_
- [Pandas Cookbook](https://github.com/jvns/pandas-cookbook/tree/master) _by Julia Evans (Code released under the CC BY-SA 4.0 DEED License)_

In [ ]:
import numpy as np
import pandas as pd

# overriding the display helper function for prettier table rendering


class display(object):
    """Display HTML representation of multiple objects"""

    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""

    def __init__(self, *args):
        self.args = args

    def _repr_html_(self):
        return "\n".join(self.template.format(a, eval(a)._repr_html_()) for a in self.args)

    def __repr__(self):
        return "\n\n".join(a + "\n" + repr(eval(a)) for a in self.args)

- Computing aggregations like `sum`, `mean`, `median`, `min`, and `max` is a fundamental operation in data science

- Pandas supports simple aggregations similar to NumPy arrays, but also allows for more sophisticated aggregations on subsets of data based on the concept of a `groupby`

## Planets Data

- For this section we will use the Planets dataset
- It gives information the 1,000+ extrasolar planets discovered up to 2014

In [ ]:
import requests
import io

planets = pd.read_csv(
    io.StringIO(requests.get("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/planets.csv").text)
)
planets.head()

## Simple Aggregation in Pandas

- We've explored some of the data aggregations available for NumPy arrays
- As with a one-dimensional NumPy array, for a Pandas ``Series`` the aggregates return a single value

In [ ]:
ser = pd.Series(np.random.rand(5))
ser

In [ ]:
ser.sum()

In [ ]:
ser.mean()

- For a `DataFrame`, by default the aggregates return results within each column

In [ ]:
df = pd.DataFrame({"A": np.random.rand(5), "B": np.random.rand(5)})
df

In [ ]:
df.mean()

- By specifying the `axis` argument, you can instead aggregate within each row

In [ ]:
df.mean(axis="columns")

- Pandas `Series` and `DataFrame` objects include all of the common aggregates
- There is also a convenience method, `describe`, that computes several common aggregates for each column

In [ ]:
planets.dropna().describe()

- `describe` helps to understand the overall properties of a dataset

----

- For example 
    - the `year` column shows that exoplanets were discovered as far back as 1989, but
    - half of all planets in the dataset were not discovered until 2010 or after
- This is largely thanks to the *Kepler* mission, which aimed to find eclipsing planets around other stars using a specially designed space telescope

- The following table summarizes some other built-in Pandas aggregations

| Aggregation              | Returns                         |
|--------------------------|---------------------------------|
| ``count``                | Total number of items           |
| ``first``, ``last``      | First and last item             |
| ``mean``, ``median``     | Mean and median                 |
| ``min``, ``max``         | Minimum and maximum             |
| ``std``, ``var``         | Standard deviation and variance |
| ``mad``                  | Mean absolute deviation         |
| ``prod``                 | Product of all items            |
| ``sum``                  | Sum of all items                |

### Exercises

1. Compute the average distance of discovered planets from our solar system
2. In what year was the planet closest to our sun discovered? (hint: `idxmin`)
3. Count the number of planets discovered with each method (hint: `value_counts`)

In [ ]:
# Solve the exercises here

In [ ]:
planets.loc[:, "distance"].mean()

In [ ]:
planets.loc[planets.loc[:, "distance"].idxmin(), "year"]

In [ ]:
planets.loc[:, "method"].value_counts()

## groupby: Split, Apply, Combine

- Often we want to aggregate conditionally on some label or index
- This is implemented in the so-called `groupby` operation
- The name "group by" comes from a command in the SQL database language, but it is perhaps easier to think of it in the terms: *split, apply, combine*

### Split, Apply, Combine

- A canonical example of this split-apply-combine operation is illustrated in this figure (apply uses sum in this example)

![](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/03.08-split-apply-combine.png?raw=1)

- The most basic split-apply-combine operation can be computed with the `groupby` method of the `DataFrame`
- We simply pass the name of the desired key column

In [ ]:
df = pd.DataFrame({"key": ["A", "B", "C", "A", "B", "C"], "data": range(6)}, columns=["key", "data"])
df

In [ ]:
df.groupby("key")

- A `DataFrameGroupBy` object is returned, and not a set of `DataFrame` objects
- You can think of it as a special view of the `DataFrame`, which defines the groups but does no actual computation until the aggregation is applied

- To produce a result, we can apply an aggregate to this `DataFrameGroupBy` object
- The aggregate will perform the appropriate apply/combine steps to produce the desired result

In [ ]:
df.groupby("key").sum()

### The GroupBy Object

- The `GroupBy` object is a flexible abstraction: in many ways, it can be treated as simply a collection of ``DataFrame``s, though it is doing more sophisticated things under the hood

#### Column indexing

- The `GroupBy` object supports column indexing in the same way as the `DataFrame`, and returns a modified `GroupBy` object

In [ ]:
planets.groupby("method")

In [ ]:
planets.groupby("method")["orbital_period"]

- Here we've selected a particular `Series` group from the original `DataFrame` group by reference to its column name
- As with the `GroupBy` object, no computation is done until we call some aggregate on the object

In [ ]:
planets.groupby("method")["orbital_period"].median()

#### Iteration over groups


- The `GroupBy` object supports direct iteration over the groups, returning each group as a `Series` or `DataFrame`

In [ ]:
for method, group in planets.groupby("method"):
    print(f"{method:30s} shape={group.shape}")

- This can be useful for manual inspection of groups for the sake of debugging
- However, it is often much faster to use the built-in `apply` functionality, which we will discuss momentarily

#### Dispatch methods

- Any method not explicitly implemented by the `GroupBy` object will be passed through and called on the groups
- For example, using the `describe` method is equivalent to calling `describe` on the `DataFrame` representing each group

In [ ]:
planets.groupby("method")["year"].describe()

- This table shows the vast majority of planets until 2014 were discovered by the Radial Velocity and Transit methods
- The newest methods seem to be Transit Timing Variation and Orbital Brightness Modulation, which were not used to discover a new planet until 2011

### Exercises

1. Which method found the farthest away / closest planets on average?
2. Which method found the planets with the largest / smallest mass on average?

In [ ]:
# Solve the exercises here

In [ ]:
planets.groupby("method")["distance"].mean().sort_values()

In [ ]:
planets.groupby("method")["mass"].mean().sort_values()

### Aggregate, Filter, Transform, Apply

- Up to now, we have only considered aggregating values in the combine step
- `GroupBy` objects provide additional `aggregate`, `filter`, `transform`, and `apply` methods that efficiently implement a variety of useful operations before combining the grouped data

In [ ]:
df = pd.DataFrame(
    {"key": ["A", "B", "C", "A", "B", "C"], "data1": range(6), "data2": np.random.randint(0, 9, 6)},
    columns=["key", "data1", "data2"],
)
df

#### Aggregation

- The `aggregate` method allows for more flexibility than simply aggregating a single value
- It can take a string, a function, or a list thereof, and compute all the aggregates at once

In [ ]:
df.groupby("key").aggregate(["min", "median", "max"])

- Another common pattern is to pass a dictionary mapping column names to operations to be applied on that column

In [ ]:
df.groupby("key").aggregate({"data1": "min", "data2": "max"})

#### Filtering

- A filtering operation allows you to drop data based on the group properties
- For example, we might want to keep all groups in which the standard deviation is larger than some value

In [ ]:
def filter_func(x):
    return x["data2"].std() > 3


display("df", "df.groupby('key').std()", "df.groupby('key').filter(filter_func)")

#### Transformation

- While aggregation must return a reduced version of the data, transformation can return some transformed version of the full data to recombine
- For such a transformation, the output is the same shape as the input
- A common example is to center the data by subtracting the group-wise mean

In [ ]:
def center(x):
    return x - x.mean()


df.groupby("key").transform(center)

#### The apply method

- The `apply` method lets you apply an arbitrary function to the group results
- The function should take a `DataFrame` and returns either a Pandas object (e.g., `DataFrame`, `Series`) or a scalar; the behavior of the combine step will be tailored to the type of output returned
- For example, here is an `apply` operation that normalizes the first column by the sum of the second

In [ ]:
def norm_by_data2(x):
    # x is a DataFrame of group values
    x["data1"] /= x["data2"].sum()
    return x


df.groupby("key")[["data1", "data2"]].apply(norm_by_data2)

### Specifying the Split Key

- In the simple examples presented before, we split the `DataFrame` on a single column name
- This is just one of many options by which the groups can be defined, and we'll go through some other options for group specification here

#### A list, array, series, or index providing the grouping keys

- The key can be any series or list with a length matching that of the `DataFrame`

In [ ]:
L = [0, 1, 0, 1, 2, 0]
df.groupby(L).sum()

- This means there's another, more verbose way of accomplishing the `df.groupby('key')` from before

In [ ]:
df.groupby(df.loc[:, "key"]).sum()

#### A dictionary or series mapping index to group

- Another method is to provide a dictionary that maps index values to the group keys

In [ ]:
df2 = df.set_index("key")
df2

In [ ]:
mapping = {"A": "vowel", "B": "consonant", "C": "consonant"}
display("df2", "df2.groupby(mapping).sum()")

#### Any Python function

- Similar to mapping, you can pass any Python function that will input the index value and output the group

In [ ]:
df2.groupby(str.lower).mean()

#### A list of valid keys

- Further, any of the preceding key choices can be combined to group on a multi-index

In [ ]:
df2.groupby([str.lower, mapping]).mean()

### Grouping Example

- As an example of this, in a few lines of Python code we can put all these together and count discovered planets by method and by decade

In [ ]:
decade = 10 * (planets.loc[:, "year"] // 10)
decade = decade.astype(str) + "s"
decade.name = "decade"
planets.groupby(["method", decade]).size().unstack().fillna(0)

### Exercises

1. Has the average distance of discovered planets changed over the years and if so, how?
2. Has the average mass of discovered planets changed over the years and if so, how?

In [ ]:
# Solve the exercises here

In [ ]:
planets.plot.scatter("year", "distance", logy=True)
planets.groupby("year")["distance"].mean()

In [ ]:
planets.plot.scatter("year", "mass")
planets.groupby("year")["mass"].mean()